In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.9 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# LiNiO₂ on ECHIDNA

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the layered oxide LiNiO₂ to constant-wavelength neutron powder data from ECHIDNA at ANSTO.
Li and Ni mix between two sites, held to full occupancy by constraints, and the profile has
Finger-Cox-Jephcoat asymmetry.

The project is the catalogue case `pd-neut-cwl_lmo-echidna`, the one the command line and the
app's Examples open. Li and Ni share their sites through five relations (displacements and
occupancies), kept from the start.

In [ ]:
import easydiffraction as edi

## Load the project

In [ ]:
project_path = edi.download_data('pd-neut-cwl_lmo-echidna', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['lmo']
experiment = project.experiments['echidna']
project.display.constraints()

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [ ]:
for parameter in project.free_parameters:
    parameter.free = False

In [ ]:
project.display.pattern('echidna')

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale, 2θ zero and cell

The scale, the zero shift and the two cell lengths put the peaks in place and at the right
height.

In [ ]:
experiment.linked_structures['lmo'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_c.free = True

In [ ]:
project.display.parameters.free()

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('echidna')

### Sample displacement and transparency

The sample sits a little off the diffractometer axis and the beam goes into it; both shift the
peaks with angle, unlike the zero.

In [ ]:
experiment.instrument.calib_sample_displacement.free = True
experiment.instrument.calib_sample_transparency.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('echidna')

### Background

The nine background points follow the level between the peaks.

In [ ]:
for point in experiment.background:
    point.intensity.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('echidna')

### Peak widths

In [ ]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('echidna')

### Oxygen position

O's height along c is the only coordinate the symmetry leaves free.

In [ ]:
structure.atom_sites['O'].fract_z.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('echidna')

### Displacements

Li1's, Ni2's and O's; Ni1 and Li2 follow by the relations.

In [ ]:
structure.atom_sites['Li1'].adp_iso.free = True
structure.atom_sites['Ni2'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('echidna')

### Li/Ni mixing

Last, Li1's occupancy, which sets how much Ni sits on the Li site; it moves with the
displacements, so it comes after them.

In [ ]:
structure.atom_sites['Li1'].occupancy.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('echidna')

In [ ]:
project.display.fit.correlations()

In [ ]:
project.display.structure('lmo')

## Save the project

In [ ]:
project.save_as('pd-neut-cwl_lmo-echidna')
project.report.save_html('pd-neut-cwl_lmo-echidna.html')